# Project 7: Intent Classification for a Smart FAQ Chatbot

**Problem:** Classify a user's message into an FAQ intent and return an appropriate predefined response. The system uses NLP classification rather than exact keyword matching.

**Workflow:** problem definition → labeled dataset → data inspection → preprocessing → TF-IDF → model comparison → evaluation → response mapping → confidence fallback → saved artifacts → Streamlit app.

## 1. Install dependencies

In [ ]:
%pip install -r requirements.txt

## 2. Load and inspect the labeled dataset

In [ ]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path('data/faq_intents.csv')
df = pd.read_csv(DATA_PATH)
print('Dataset shape:', df.shape)
print('\nMissing values:\n', df.isna().sum())
print('Duplicate messages:', df['text'].duplicated().sum())
display(df.head(10))
display(df['intent'].value_counts().sort_index().to_frame('count'))

## 3. Exploratory data analysis

In [ ]:
import matplotlib.pyplot as plt

ax = df['intent'].value_counts().plot(kind='bar', figsize=(12, 4), title='Samples per intent')
ax.set_xlabel('Intent')
ax.set_ylabel('Number of messages')
plt.xticks(rotation=70, ha='right')
plt.tight_layout()
plt.show()

## 4. Preprocessing decisions

The training script trims whitespace and removes missing or duplicate records. TF-IDF lowercases text by default, tokenizes it, and creates word and character n-gram features. Negation words are not removed blindly because they may change the intent. Character n-grams can help with minor spelling variation.

## 5. Train and compare two classification models

In [ ]:
%run train_model.py

## 6. Review evaluation results

In [ ]:
results_path = Path('reports/model_comparison.csv')
if results_path.exists():
    display(pd.read_csv(results_path))
else:
    print('Run the training cell first.')

errors_path = Path('reports/misclassified_examples.csv')
if errors_path.exists():
    display(pd.read_csv(errors_path).head(30))

## 7. Test an example message

In [ ]:
import json, joblib

model = joblib.load('models/smart_faq_pipeline.joblib')
responses = json.loads(Path('models/faq_responses.json').read_text(encoding='utf-8'))

def predict_faq(message, threshold=0.60):
    if not isinstance(message, str) or not message.strip():
        return {'intent': None, 'confidence': 0.0, 'response': 'Please enter a question.'}
    probs = model.predict_proba([message])[0]
    idx = int(probs.argmax())
    intent, confidence = str(model.classes_[idx]), float(probs[idx])
    if confidence < threshold:
        answer = 'I am not confident enough to identify your question. Please rephrase it or contact the relevant office.'
    else:
        answer = responses.get(intent, 'Please contact the relevant office.')
    return {'intent': intent, 'confidence': round(confidence, 3), 'response': answer}

for question in ['How do I apply for admission?', 'How can I get my fee receipt?', 'I forgot my password', 'What is the college phone number?', 'Can you recommend a movie?']:
    print('Question:', question)
    print(predict_faq(question))
    print()

## 8. Launch the Streamlit application

Run from a terminal in the project directory:

```bash
streamlit run app.py
```

The interface includes clickable sample questions, a chat history, predefined responses, prediction details, and a confidence-based fallback.

## 9. Limitations and conclusion

This project implements a complete baseline NLP intent-classification workflow and compares two models on held-out data. The sample dataset is small and custom-created, so evaluation on new, real user queries is necessary. Institution-specific deadlines, fees, office hours, and contact details should be filled in only from verified sources. Intent classification predicts the category of a message; the chatbot then selects a stored response rather than generating a new answer.